# LLM From Scratch

## Transformer Implementation

### Importing Libraries

In [1]:
from dataclasses import dataclass
from functools import wraps
from typing import List

import nltk
import numpy as np
import torch
import torchvision
import tqdm
from torch import nn, optim
from torchvision import transforms

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
torch.manual_seed(42)

### Transformer and Vision Transformer

In [11]:


class AverageMeter:

    def __init__(self):
        self.num = 0
        self.tot = 0

    def update(self, val: float, sz: float):
        self.num += val * sz
        self.tot += sz

    def calculate(self) -> float:
        return self.num / self.tot


def averager(func):
    num = 0
    total = 0

    @wraps(func)
    def inner(val, sz):
        nonlocal num, total
        num += val * sz
        total += sz
        avg = num / total if total > 0 else 0.0
        return func(avg)

    return inner


In [16]:
loss = 0
acc = 0


@averager
def log_loss(x):
    print(f"Current average: {x:4f}")


log_loss(2, 3)
log_loss(3, 4)


Current average: 2.000000
Current average: 2.571429


In [42]:



@dataclass(init=True)
class TrainResult:
    """
    A collection containing everything we need to know about the training results
    """
    train_losses: List[float]
    train_accs: List[float]
    val_accs: List[float]
    val_losses: List[float]


device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'


class AverageMeter:

    def __init__(self):
        self.num = 0
        self.tot = 0

    def update(self, val: float, sz: float):
        self.num += val * sz
        self.tot += sz

    def calculate(self) -> float:
        return self.num / self.tot


def averager(func):
    num = 0
    total = 0

    def update(val, sz):
        nonlocal num
        nonlocal total
        num += val * sz
        total += sz

    @wraps(func)
    def inner():
        nonlocal num, total
        update(num / total)

    return inner


def print_variance(name, data):
    neuron_variance = torch.mean(torch.var(data, dim=0))
    print(f'name={name!r}, Variance={neuron_variance.float()}')


class MultiAttentionHead(nn.Module):

    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        super().__init__()
        self.H = num_heads
        self.n_hidden = n_hidden
        self.qkv = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        self.WO = nn.Linear(num_heads * n_hidden, dim)

    def forward(self, x: torch.Tensor, attn_mask=None) -> tuple[torch.Tensor, torch.Tensor]:
        """
                attention mask
                masks future inputs

        that those blank tokens get filled in with the prediction

        """
        batch_size, token_size, dim = x.shape
        QKV = self.qkv(x)
        QKV = QKV.reshape(batch_size, token_size, self.H * self.n_hidden * 3)
        QKV = QKV.transpose(-2, -1)
        Q, K, V = QKV.chunk(3, dim=-1)
        A = Q @ K.transpose(-2, -1) / self.n_hidden ** 0.5
        scores = A
        if attn_mask is not None:
            '\n            causal masking future inputs of 0 to negative so that softmax will return 0\n            '
            scores = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float('-inf'))
        A = torch.softmax(scores, dim=-1)
        Z = A @ V
        Z = Z.transpose(-2, -1)
        Z = Z.reshape(batch_size, token_size, self.H * self.n_hidden)
        Z = self.WO(Z)
        return (Z, A)


import torch
from torch import nn


class MultiHeadedAttention(nn.Module):

    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden
        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        # Projection Head
        self.W0 = nn.Linear(num_heads * n_hidden, dim)

    @staticmethod
    def create_positional_encoding():
        #todo
        pass

    def forward(self, x: torch.Tensor, attn_mask=None) -> tuple[torch.Tensor, torch.Tensor]:
        B, T, dim = x.shape
        qkv = self.qkv_projection(x)
        qkv = qkv.reshape(B, T, self.num_heads, 3 * self.n_hidden).transpose(1, 2)
        q, k, v = qkv.chunk(3, dim=-1)
        scores = torch.matmul(q, k.transpose(-2, -1)) / self.n_hidden ** 0.5
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float('-inf'))
        attn_alphas = torch.softmax(scores, dim=-1)
        context = attn_alphas @ v
        A = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)
        Z = self.W0(A)
        return (Z, attn_alphas)


class AttentionResidual(nn.Module):

    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        super().__init__()
        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)
        self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, mlp_dim),
                                 nn.LayerNorm(dim),
                                 nn.GELU(),  # x*phi(x)-> CDF of x applied element wise
                                 nn.Linear(mlp_dim, dim))
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor, attn_mask=False) -> tuple[torch.Tensor, torch.Tensor]:
        #print_variance('Input', x)
        Z, A = self.attn(x, attn_mask=attn_mask)
        print_variance('after first attention block output', Z)
        x = Z + x
        #print_variance('after residual 1 adding output to Z', x)
        ffn_out = self.ffn(x)
        x = ffn_out + x
        print_variance('after Applying FeedForward output with residual', x)
        #print('Residual Block\n\n')
        return (x, A)


class Transformer(nn.Module):

    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int):
        super().__init__()
        self.layers = nn.ModuleList([AttentionResidual(dim, attn_dim, mlp_dim, num_heads) for _ in range(num_layers)])

    def forward(self, x: torch.Tensor, attn_mask=False, return_attn=False) -> tuple[torch.Tensor, torch.Tensor | None]:
        """
        # x                the inputs. shape: (B x T x dim)
        # attn_mask        an attention mask. Pass this to each of the AttentionResidual layers!
        #                  shape: (B x T x T)
        #
        # Outputs:
        # attn_output      shape: (B x T x dim)
        # attn_alphas      If return_attn is False, return None. Otherwise return the attention weights
        #                  of each of each of the attention heads for each of the layers.
        #                  shape: (B x Num_layers x Num_heads x T x T)
        """
        output = None
        collected_attns = []
        Z = None
        A = []
        #print(f'\n\n-------------------------Transformer: {x.shape}')
        for residual in self.layers:
            x, alphas = residual(x, attn_mask=attn_mask)
            if return_attn is not None:
                A.append(alphas)
        if return_attn:
            return (x, torch.stack(A, dim=1))
        else:
            return (x, None)


class PatchEmbed(nn.Module):
    """Image to Patch Embedding"""

    def __init__(self, img_size: int, patch_size: int, nin: int, nout: int):
        super().__init__()
        assert img_size % patch_size == 0
        self.img_size = img_size
        self.flattened_patch = (img_size // patch_size) ** 2
        self.patch_embedding = nn.Conv2d(in_channels=nin, out_channels=nout, kernel_size=patch_size, stride=patch_size)
        self.nout = nout

    def forward(self, x: torch.Tensor):
        """TODO: Implement the patch embedding. You want to split up the image into square patches of the given patch size. Then each patch_size x patch_size square should be linearly projected into an embedding of size nout. Hint: Take a look at nn.Conv2d. How can this be used to perform the patch embedding?"""
        out = self.patch_embedding(x)
        out = out.flatten(2)
        out = out.transpose(1, 2).long()
        print(f'Output shape in patch embedding: out.shape={out.shape!r}')
        return out


class VisionTransformer(nn.Module):

    def __init__(self, n_channels: int, nout: int, img_size: int, patch_size: int, dim: int, attn_dim: int,
                 mlp_dim: int, num_heads: int, num_layers: int):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim)
        self.pos_E = nn.Embedding((img_size // patch_size) ** 2, dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.transformer = Transformer(dim=dim, attn_dim=attn_dim, mlp_dim=mlp_dim, num_heads=num_heads,
                                       num_layers=num_layers)
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, nout))

    def forward(self, img: torch.Tensor, return_attn=False) -> tuple[torch.Tensor, torch.Tensor | None]:
        embs = self.patch_embed(img)
        B, T, _ = embs.shape
        print(f"{embs.shape=}")
        pos_ids = torch.arange(T).expand(B, -1).to(embs.long().to(device))
        embs = embs + self.pos_E(pos_ids).long()
        print_variance('Embeddings', embs.float())
        cls_token = self.cls_token.expand(len(embs), -1, -1)

        x = torch.cat([cls_token, embs], dim=1)
        print(f"{x.shape=}")
        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)
        print_variance('Output', x)
        out = self.head(x)[:, 0]
        print_variance('Projected Output', x)
        return (out, alphas)


@torch.compile
def evaluate_cifar_model(model, criterion, val_loader):
    is_train = model.training
    model.eval()
    with torch.no_grad():
        loss_meter, acc_meter = (AverageMeter(), AverageMeter())
        for img, labels in val_loader:
            img = img.to(device)
            labels = labels.to(device)
            outputs, _ = model(img)
            loss_meter.update(criterion(outputs, labels).item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
    model.train(is_train)
    return (loss_meter.calculate(), acc_meter.calculate())



In [106]:
img = torchvision.io.read_image('flower.jpeg')

In [107]:
img.shape

torch.Size([3, 336, 224])

In [108]:
img.size()

torch.Size([3, 336, 224])

In [43]:
MEAN = [0.4914, 0.4822, 0.4465]
STD = [0.247, 0.2435, 0.2616]
batch_size = 256
img_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=MEAN, std=STD)])

inv_transform = transforms.Compose([transforms.Normalize(mean=[0.0, 0.0, 0.0], std=1 / np.array(STD)),
                                    transforms.Normalize(mean=-np.array(MEAN), std=[1.0, 1.0, 1.0]),
                                    transforms.ToPILImage()])




In [44]:
train_dataset = torchvision.datasets.CIFAR10(train=True, root='data', transform=img_transform, download=True)
val_dataset = torchvision.datasets.CIFAR10(train=False, root='data', transform=img_transform)

In [46]:
img, label = train_dataset[0]

RuntimeError: Boolean value of Tensor with more than one value is ambiguous

In [49]:
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=1)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=1)

In [51]:
x = torch.randn(3).to(device)
y = torch.randn(3)

In [52]:
@torch.compile
def foo(x):
    return torch.sin(x) + torch.cos(x)




In [53]:
%timeit foo(x)

8.12 μs ± 24.9 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [54]:
%timeit foo(y)

The slowest run took 4.81 times longer than the fastest. This could mean that an intermediate result is being cached.
20.4 μs ± 15.2 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [59]:
%%timeit
y = torch.compile(foo)
y(torch.randn(100).to(device))


409 μs ± 9.51 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [61]:
%timeit foo(torch.randn(100).to(device))


191 μs ± 1.23 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [40]:

y = torch.compile(foo)

In [41]:
y(x)

tensor([-0.4682,  0.7974,  0.2085], device='mps:0')

In [17]:
model = VisionTransformer(n_channels=3, nout=10, img_size=32, patch_size=4, dim=128, attn_dim=64, mlp_dim=128,
                          num_heads=3, num_layers=6).to(device)

NameError: name 'VisionTransformer' is not defined

In [ ]:
def train(model, data, epochs):

In [109]:
def main():
    model = VisionTransformer(n_channels=3, nout=10, img_size=32, patch_size=4, dim=128, attn_dim=64, mlp_dim=128,
                              num_heads=3, num_layers=6).to(device)
    MEAN = [0.4914, 0.4822, 0.4465]
    STD = [0.247, 0.2435, 0.2616]
    img_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=MEAN, std=STD)])
    inv_transform = transforms.Compose([transforms.Normalize(mean=[0.0, 0.0, 0.0], std=1 / np.array(STD)),
                                        transforms.Normalize(mean=-np.array(MEAN), std=[1.0, 1.0, 1.0]),
                                        transforms.ToPILImage()])
    train_dataset = torchvision.datasets.CIFAR10(train=True, root='data', transform=img_transform, download=True)
    val_dataset = torchvision.datasets.CIFAR10(train=False, root='data', transform=img_transform)
    train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=1)
    val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=1)

    criterion = nn.CrossEntropyLoss()

    NUM_EPOCHS = 10

    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.003)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])

    for epoch in range(NUM_EPOCHS):
        loss_meter = AverageMeter()
        acc_meter = AverageMeter()
        for img, labels in tqdm.tqdm(train_dataloader, desc='Training at ' + str(epoch)):
            img, labels = (img.to(device), labels.to(device))
            optimizer.zero_grad()
            outputs, _ = model(img)
            loss = criterion(outputs, labels)
            loss_meter.update(loss.item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
            loss.backward()
            optimizer.step()

        scheduler.step()
        result.train_losses.append(loss_meter.calculate())
        result.train_accs.append(acc_meter.calculate())
        print(f'Train Epoch: {epoch}, Loss: {loss_meter.calculate()}, Acc: {acc_meter.calculate()}')
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        result.val_losses.append(val_loss)
        result.val_accs.append(val_acc)
        print(f'Val Epoch: {epoch}, Loss: {val_loss}, Acc: {val_acc}')
        if epoch > 0 and result.val_losses[epoch] > result.val_losses[epoch - 1]:
            torch.save(model.state_dict(), 'Vision_transformer_Best_' + str(epoch + 1) + '.pt')
            val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
            print(f'Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}')
            print('Finished Training')
            break
        else:
            torch.save(model.state_dict(), 'Vision_transformer_' + str(epoch + 1) + '.pt')
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        print(f'Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}')
        print('Finished Training')


In [4]:
device

'mps'

In [5]:
import sys

sys.version_info

sys.version_info(major=3, minor=14, micro=3, releaselevel='final', serial=0)

## GPT

### Tokenization

In [46]:
with open('shakespeare.txt') as f:
    data = f.read()

In [47]:
corpus = data.split("\n\n")

In [51]:
from pprint import pprint

pprint(corpus[:10])
print(corpus[:10])

['First Citizen:\nBefore we proceed any further, hear me speak.',
 'All:\nSpeak, speak.',
 'First Citizen:\nYou are all resolved rather to die than to famish?',
 'All:\nResolved. resolved.',
 'First Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.',
 "All:\nWe know't, we know't.",
 'First Citizen:\n'
 "Let us kill him, and we'll have corn at our own price.\n"
 "Is't a verdict?",
 "All:\nNo more talking on't; let it be done: away, away!",
 'Second Citizen:\nOne word, good citizens.',
 'First Citizen:\n'
 'We are accounted poor citizens, the patricians good.\n'
 'What authority surfeits on would relieve us: if they\n'
 'would yield us but the superfluity, while it were\n'
 'wholesome, we might guess they relieved us humanely;\n'
 'but they think we are too dear: the leanness that\n'
 'afflicts us, the object of our misery, is as an\n'
 'inventory to particularise their abundance; our\n'
 'sufferance is a gain to them Let us revenge this with\n'
 'our pikes, ere we be

In [19]:
for line in data[:100]:
    print(line)

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [26]:
from collections import Counter
from nltk.tokenize import word_tokenize

nltk.download("punkt")

[nltk_data] Downloading package punkt to /Users/deven/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [21]:
vocab = Counter(data)

In [22]:
list(vocab.keys())[:100]

['First Citizen:',
 'Before we proceed any further, hear me speak.',
 '',
 'All:',
 'Speak, speak.',
 'You are all resolved rather to die than to famish?',
 'Resolved. resolved.',
 'First, you know Caius Marcius is chief enemy to the people.',
 "We know't, we know't.",
 "Let us kill him, and we'll have corn at our own price.",
 "Is't a verdict?",
 "No more talking on't; let it be done: away, away!",
 'Second Citizen:',
 'One word, good citizens.',
 'We are accounted poor citizens, the patricians good.',
 'What authority surfeits on would relieve us: if they',
 'would yield us but the superfluity, while it were',
 'wholesome, we might guess they relieved us humanely;',
 'but they think we are too dear: the leanness that',
 'afflicts us, the object of our misery, is as an',
 'inventory to particularise their abundance; our',
 'sufferance is a gain to them Let us revenge this with',
 'our pikes, ere we become rakes: for the gods know I',
 'speak this in hunger for bread, not in thirst for

In [11]:
vocabulary = {}

['__annotate__',
 '__annotations__',
 '__builtins__',
 '__call__',
 '__class__',
 '__closure__',
 '__code__',
 '__defaults__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__get__',
 '__getattribute__',
 '__getstate__',
 '__globals__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__kwdefaults__',
 '__le__',
 '__lt__',
 '__module__',
 '__name__',
 '__ne__',
 '__new__',
 '__qualname__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__type_params__']

In [23]:
def tokenize(sentence):
    return word_tokenize(sentence)

In [24]:
word_tokenize(data[0])

['First', 'Citizen', ':']

In [37]:
def tokenizer(data):
    out = np.array()
    for line in data:
        out += "<cls>" + word_tokenize(line)
        word_tokenize(line)


In [38]:
tokens = tokenizer(data)

TypeError: can only concatenate str (not "list") to str

In [76]:
vocab = np.unique(tokenize(data))
vocab = np.concatenate([np.array(["<start>", "<pad>", "<unk>"]), vocab])


In [83]:
tokens_to_id = {w: i for i, w in enumerate(vocab)}

In [84]:
vocab_size = len(vocab)

In [85]:
vocab[:100]

array(['<start>', '<pad>', '<unk>', '!', '$', '&', "'", "'d", "'ll", "'m",
       "'re", "'s", "'t", "'ve", ',', '--', '.', '3', ':', ';', '?', 'A',
       'ABHORSON', 'ABRAHAM', 'ADRIAN', 'AEacides', 'AEdile', 'AEdiles',
       'AEneas', 'AEsop', 'ALL', 'ALONSO', 'ANGELO', 'ANNE', 'ANOTHER',
       'ANTIGONUS', 'ANTONIO', 'ARCHBISHOP', 'ARCHIDAMUS', 'ARIEL',
       'AUFIDIUS', 'AUMERLE', 'AUTOLYCUS', 'Abase', 'Abate', 'Abated',
       'Abbot', 'Abel', 'Abhorred', 'Abhorson', 'Abides', 'Able', 'About',
       'Above', 'Abraham', 'Absolute', 'Accept', 'Accomplish',
       'According', 'Accords', 'Account', 'Accountant', 'Accursed',
       'Accuse', 'Achieve', 'Acquaint', 'Action', 'Adam', 'Add', 'Added',
       'Adding', 'Address', 'Adieu', 'Adjudged', 'Admit', 'Adonis',
       'Adoptedly', 'Adopts', 'Adrian', 'Adriatic', 'Advance',
       'Advantaging', 'Adversity', 'Advertising', 'Advocate', 'Affection',
       'Affliction', 'Affrighted', 'Affrights', 'Affront', 'Afore',
       'Afres

In [92]:
input_string = "KING RICHARD III:\nSay that I did all this for love of her."
input_string.split("\n")

['KING RICHARD III:', 'Say that I did all this for love of her.']

In [95]:
for token in input_string.split():

    if token in tokens_to_id:
        id_tensor = tokens_to_id[tokens]



KeyError: 'III:'

In [86]:
torch.nn.utils.rnn.pad_sequence(

RuntimeError: pad_sequence: Expected iterable for input sequences, but got arg of type: <class 'int'>

In [72]:
def encode(s: str) -> torch.Tensor:
    """

    Args:
        s: Input String

    Returns:
        id_tensor   a tensor of token ids, starting with the start token.t

    """


['<start>' '<pad><unk>' '!' '$' '&' "'" "'d" "'ll" "'m" "'re" "'s" "'t"
 "'ve" ',' '--' '.' '3' ':' ';' '?' 'A' 'ABHORSON' 'ABRAHAM' 'ADRIAN'
 'AEacides' 'AEdile' 'AEdiles' 'AEneas' 'AEsop' 'ALL' 'ALONSO' 'ANGELO'
 'ANNE' 'ANOTHER' 'ANTIGONUS' 'ANTONIO' 'ARCHBISHOP' 'ARCHIDAMUS' 'ARIEL'
 'AUFIDIUS' 'AUMERLE' 'AUTOLYCUS' 'Abase' 'Abate' 'Abated' 'Abbot' 'Abel'
 'Abhorred' 'Abhorson' 'Abides' 'Able' 'About' 'Above' 'Abraham'
 'Absolute' 'Accept' 'Accomplish' 'According' 'Accords' 'Account'
 'Accountant' 'Accursed' 'Accuse' 'Achieve' 'Acquaint' 'Action' 'Adam'
 'Add' 'Added' 'Adding' 'Address' 'Adieu' 'Adjudged' 'Admit' 'Adonis'
 'Adoptedly' 'Adopts' 'Adrian' 'Adriatic' 'Advance' 'Advantaging'
 'Adversity' 'Advertising' 'Advocate' 'Affection' 'Affliction'
 'Affrighted' 'Affrights' 'Affront' 'Afore' 'Afresh' 'Afric' 'African'
 'After' 'Again' 'Against' 'Agamemnon' 'Age' 'Aged' 'Agenor' 'Agreed'
 'Agrippa' 'Ah' 'Aim' 'Aiming' 'Airy' 'Ajax' "Al'ce" 'Alack' 'Alas'
 'Alban' 'Albeit' 'Albion' '

In [ ]:
vocab = np.array()
for line in data[:10]:
    vocab = np.unique("<sos>" + " ".join(word_tokenize(line)) + "</eos>")
    np.concat()

In [100]:
from typing import Optional


class Tokenizer:
    def __init__(self, corpus: Optional[str] = None):
        self.start = "<cls>"
        self.padding = "<pad>"
        self.unknown = "<unk>"
        self.vocab = np.unique(tokenize(data))
        self.vocab = np.concatenate([np.array([self.start, self.padding, self.unknown]), vocab])
        self.tokens_to_id = {w: i for i, w in enumerate(self.vocab)}
        self.vocab_size = len(self.vocab)

    def encode(self, s: str) -> torch.Tensor:
        """

        Args:
            s: Input String

        Returns:
            id_tensor   a tensor of token ids, starting with the start token.t

        """
        #self.vocab = np.concatenate([np.array([self.start, self.padding, self.unknown]), np.unique(s)])
        #self.tokens_to_id = {w: i for i, w in enumerate(self.vocab)}

    def decode(self, toks: torch.Tensor) -> str:
        # toks         a list of token ids
        #
        # Output
        # decoded_str  the token ids decoded back into a string (join with a space)

        decoded_str = " ".join(toks)

        # TODO: convert the token ids back to the actual corresponding words.
        # Join the tokens with a space and return the full string
        # ============ ANSWER START ===========

        # ============ ANSWER END =============

        return decoded_str

    def pad_examples(self, tok_list: List[torch.Tensor]) -> torch.Tensor:
        return torch.nn.utils.rnn.pad_sequence(
            tok_list, batch_first=True, padding_value=self.tokens_to_id[self.padding]
        )







In [101]:
tokenizer = Tokenizer()

In [103]:
input_string

'KING RICHARD III:\nSay that I did all this for love of her.'